In [27]:
import snowflake.connector
import os
import re
from datetime import datetime
import pandas as pd

def get_incremental(table, schema, columns, column_definitions):
    return """
/**********************************
File built from dbt_code_generator.ipynb
model file generated on: """+str(datetime.now())+"""
***********************************/

{{ config(
    materialized='incremental',
    incremental_strategy='merge',
    alias='"""+table+"""',
    unique_key = [''],
    transient=false,
) }}

with source_data as
(
    SELECT
        """+', \n\t\t'.join(columns)+"""
    FROM {{ source('"""+schema+"""','"""+table.upper()+"""') }}
), 

transform_data as 
(
    SELECT
        """+', \n\t\t'.join(columns)+""",
        '{{ invocation_id }}' AS AUDIT_LOG_ID,
        SHA2('hash_me') as _ROW_HASH -- Adds column to table on full refresh.
        FROM source_data
), 

type_data as
(
    SELECT
        """+', \n\t\t'.join(['CAST('+column+' as '+column_definitions[column]+') as '+column for column in columns])+""",
        CAST(AUDIT_LOG_ID as VARCHAR(128)) as AUDIT_LOG_ID,
        CAST(_ROW_HASH as VARCHAR(128)) as _ROW_HASH
    FROM transform_data
)

SELECT * FROM type_data
"""


def get_view(table, schema, columns):
    return """
/**********************************
File built from dbt_code_generator.ipynb
model file generated on: """+str(datetime.now())+"""
***********************************/

{{ config(
    materialized='view', 
    alias='"""+table+"""', 
    transient=false
) }}

with source_data as
(   
    SELECT
        """+', \n\t\t'.join(columns)+"""
    FROM {{ source('"""+schema+"""','"""+table.upper()+"""') }}
), 

transform_data as
(
 SELECT * FROM source_data
)

SELECT * FROM transform_data
"""



In [28]:
database = 'COMMON'
schema = 'SAPO'
project = 'common_to_pub'
folder = 'sapo'

path = 'C:\\Sources\\Repos\\EnterpriseCloud\\dbt\\'+project+'\\models\\'+folder

info_conn = snowflake.connector.connect(
    user = 'jay.spiers@us.tel.com',
    authenticator = 'externalbrowser',
    account = 'telus-global.privatelink',
    warehouse = 'DEVELOPER_WH',
    database = database,
    schema = 'INFORMATION_SCHEMA'
)

info_curs = info_conn.cursor()
info_curs.execute('USE ROLE SYSADMIN_DEV;')

schema_conn = snowflake.connector.connect(
    user = 'jay.spiers@us.tel.com',
    authenticator = 'externalbrowser',
    account = 'telus-global.privatelink',
    warehouse = 'DEVELOPER_WH',
    database = database,
    schema = schema
)

schema_curs = schema_conn.cursor()
schema_curs.execute('USE ROLE SYSADMIN_DEV;')

In [54]:
def gen_incrementals(path, info_curs, schema_curs, schema):
    
    prebuilt_models = [file[:-4] for file in os.listdir(path) if file[-4:]=='.SQL']

    tables_df = info_curs.execute("SELECT TABLE_NAME FROM TABLES WHERE TABLE_SCHEMA = '"+schema+"' ORDER BY TABLE_NAME ASC").fetch_pandas_all()
    table_names = [name for name in list(tables_df['TABLE_NAME'])]

    for table in table_names[:1]:
        if table in prebuilt_models:
            file = open(path+'\\'+table+'.sql', 'r')
            if 'xxxxxxxxxxxxxxxxxxFile built from dbt_code_generator.py' not in file.read():
                continue
            file.close()
        ddl = schema_curs.execute("SELECT get_ddl('table', '"+table+"');").fetchone()[0]
        ddl_stripped = re.search('(?s)create or replace TABLE '+table+' \(\n\t(.*)\)(?:COMMENT=|;)', ddl).groups()[0]
        ddl_split = re.split(',\n\t', ddl_stripped)

        column_definitions = {d.split(' ')[0]:' '.join(d.split(' ')[1:]) for d in ddl_split if d.split(' ')[0] not in ['AUDIT_LOG_ID', '_ROW_HASH']}
        columns = list(column_definitions)

        print(get_incremental(table, schema, columns, column_definitions))

        file_name = path+'\\'+table+'.SQL'
        # dbt_model = open(file_name, 'w')
        # dbt_model.write(get_incremental(table, columns, column_definitions))
        # dbt_model.close()
        print(file_name)

def gen_views(path, info_curs, schema_curs, schema):
    
    prebuilt_models = [file[:-4] for file in os.listdir(path) if file[-4:]=='.SQL']

    tables_df = info_curs.execute("SELECT TABLE_NAME FROM TABLES WHERE TABLE_SCHEMA = '"+schema+"' ORDER BY TABLE_NAME ASC").fetch_pandas_all()
    table_names = [name for name in list(tables_df['TABLE_NAME'])]

    yml = ""
    yml_file = open(path+'\\schema.yml', 'r')
    old_yml = yml_file.read()
    yml_file.close()

    for table in table_names:
        if table in prebuilt_models:
            file = open(path+'\\'+table+'.sql', 'r')
            if 'xxxxxxxxxxxxxxxxxxFile built from dbt_code_generator.py' not in file.read():
                continue
            file.close()
        ddl = schema_curs.execute("SELECT get_ddl('table', '"+table+"');").fetchone()[0]
        ddl_stripped = re.search('(?s)create or replace TABLE '+table+' \(\n\t(.*)\)(?:COMMENT=|;)', ddl).groups()[0]
        ddl_split = re.split(',\n\t', ddl_stripped)

        column_definitions = {d.split(' ')[0]:' '.join(d.split(' ')[1:]) for d in ddl_split if d.split(' ')[0] not in ['AUDIT_LOG_ID', '_ROW_HASH']}
        columns = list(column_definitions)

        sql = get_view(table, 'COMMON_SAPO', columns)

        file_name = path+'\\v_'+table+'.SQL'
        dbt_model = open(file_name, 'w')
        dbt_model.write(sql)
        dbt_model.close()
        
        if '- name: '+table not in old_yml:
            yml += """
      - name: """+table+"""
        identifier: """+table+"""
        description:  Model SAPO """+table+""" file."""
            
        print(file_name)

    yml_file = open(path+'\\schema.yml', 'w')
    yml_file.write(old_yml+yml)
    yml_file.close()
    print('Completed succesfully.')

gen_views(path, info_curs, schema_curs, schema)

C:\Sources\Repos\EnterpriseCloud\dbt\common_to_pub\models\sapo\v_ABAP_4_VARIANT_STORAGE.SQL
C:\Sources\Repos\EnterpriseCloud\dbt\common_to_pub\models\sapo\v_ASSIGNMENT_IM_STRG_LOC_TO_WM_WAREHOUSE.SQL
C:\Sources\Repos\EnterpriseCloud\dbt\common_to_pub\models\sapo\v_ASSIGNMENT_TASK_LISTS_MATERIALS.SQL
C:\Sources\Repos\EnterpriseCloud\dbt\common_to_pub\models\sapo\v_BILLING_DOCUMENT_HEADER_DATA.SQL
C:\Sources\Repos\EnterpriseCloud\dbt\common_to_pub\models\sapo\v_BILLING_DOCUMENT_ITEM_DATA.SQL
C:\Sources\Repos\EnterpriseCloud\dbt\common_to_pub\models\sapo\v_CATALOG_TEXTS.SQL
C:\Sources\Repos\EnterpriseCloud\dbt\common_to_pub\models\sapo\v_CODE_GROUP_TEXTS.SQL
C:\Sources\Repos\EnterpriseCloud\dbt\common_to_pub\models\sapo\v_CODE_TEXTS.SQL
C:\Sources\Repos\EnterpriseCloud\dbt\common_to_pub\models\sapo\v_COMPANY_CODES.SQL
C:\Sources\Repos\EnterpriseCloud\dbt\common_to_pub\models\sapo\v_CONTROL_PARAMETERS_FOR_MRP.SQL
C:\Sources\Repos\EnterpriseCloud\dbt\common_to_pub\models\sapo\v_CUSTOMER_MAS

In [37]:
prebuilt_models = [file[:-4] for file in os.listdir(path) if file[-4:]=='.SQL']

tables_df = info_curs.execute("SELECT TABLE_NAME FROM TABLES WHERE TABLE_SCHEMA = '"+schema+"' ORDER BY TABLE_NAME ASC").fetch_pandas_all()
table_names = [name for name in list(tables_df['TABLE_NAME'])]

yml = ""

for table in table_names[:2]:
    if table in prebuilt_models:
        file = open(path+'\\'+table+'.sql', 'r')
        if 'xxxxxxxxxxxxxxxxxxFile built from dbt_code_generator.py' not in file.read():
            continue
        file.close()
    ddl = schema_curs.execute("SELECT get_ddl('table', '"+table+"');").fetchone()[0]
    ddl_stripped = re.search('(?s)create or replace TABLE '+table+' \(\n\t(.*)\)(?:COMMENT=|;)', ddl).groups()[0]
    ddl_split = re.split(',\n\t', ddl_stripped)

    column_definitions = {d.split(' ')[0]:' '.join(d.split(' ')[1:]) for d in ddl_split if d.split(' ')[0] not in ['AUDIT_LOG_ID', '_ROW_HASH']}
    columns = list(column_definitions)

    sql = get_view(table, 'COMMON_SAPO', columns)
    yml += """
      - name: """+table+"""
        identifier: """+table+"""
        description:  Model SAPO """+table+""" file."""
print(yml)



      - name: ABAP_4_VARIANT_STORAGE
        identifier: ABAP_4_VARIANT_STORAGE
        description:  Model SAPO ABAP_4_VARIANT_STORAGE file.
      - name: ASSIGNMENT_IM_STRG_LOC_TO_WM_WAREHOUSE
        identifier: ASSIGNMENT_IM_STRG_LOC_TO_WM_WAREHOUSE
        description:  Model SAPO ASSIGNMENT_IM_STRG_LOC_TO_WM_WAREHOUSE file.


In [ ]:
project = 'lz_to_ods'
schema = 'ITHD'
db = 'ODS'
path = "C:\\Users\\223749\\Downloads\\ithd schema.csv"


template = """
{{ config(
    materialized='incremental',
    incremental_strategy='merge',
    alias='<<TABLE_NAME>>',
    unique_key=['ID'],
    transient=false
) }} 

with source_data as (
    SELECT
        _FILEDATE,
        _FILENAME,
        _UPLOADDATE,
        <<SOURCE>>
    FROM {{ source('<<SCHEMA_NAME>>', '<<TABLE_NAME>>') }}
    {% if is_incremental() %}WHERE _UPLOADDATE > IFNULL((SELECT MAX(_UPLOADDATE) FROM {{ this }}),'1970-01-01'){% endif %}
    QUALIFY (ROW_NUMBER() OVER (PARTITION BY ID ORDER BY _FILEDATE DESC, _UPLOADDATE DESC, MODIFIEDON DESC)) = 1
),

transform_data as (
    SELECT
        _FILEDATE,
        _FILENAME,
        _UPLOADDATE,
        '{{ invocation_id }}' AS AUDIT_LOG_ID,
        SHA2('hash_me') AS _ROW_HASH,
        <<TRANSFORM>>
    FROM source_data
),

type_data as (
    SELECT 
        _FILEDATE,
        _FILENAME,
        _UPLOADDATE,
        AUDIT_LOG_ID,
        _ROW_HASH,
        <<TYPE>>
    FROM transform_data
)

select * from type_data
"""


yml = ''

snow = f"""
use database <<{db}>>;
use schema {schema};

"""

housekeeping_fields = """
    _FILEDATE TIMESTAMP_NTZ(9),
    _FILENAME VARCHAR(16777216),
    _UPLOADDATE TIMESTAMP_NTZ(9),
    AUDIT_LOG_ID VARCHAR(36),
    _ROW_HASH VARCHAR(128),
"""

df = pd.read_csv(path)
for table in sorted(set(df['Table'])):
    df_filt = df[df['Table']==table]
    cols = df_filt['Column']
    cols_upper = cols.str.upper()
    dtypes = df_filt['SF Type']

    source = 'JSONFIELD:'+cols+'::STRING as '+cols_upper
    source_str = ',\n        '.join(source)
    transform = cols_upper
    transform_str = ',\n        '.join(transform)
    type = 'cast('+cols_upper+' as '+dtypes+') as '+cols_upper
    type_str = ',\n        '.join(type)

    dbt = template.replace('<<TABLE_NAME>>', table.upper())
    dbt = dbt.replace('<<SCHEMA_NAME>>', schema.upper())
    dbt = dbt.replace('<<SOURCE>>', source_str)
    dbt = dbt.replace('<<TRANSFORM>>', transform_str)
    dbt = dbt.replace('<<TYPE>>', type_str)

    yml += f"  - name: {table.upper()}\n    identifier: {table.upper()}\n    description: Landing table for {table.upper()} file.\n"

    s = ",\n    ".join(cols_upper + " " + dtypes)
    snow += "create or replace TABLE " + table.upper() +" (" + housekeeping_fields + "    " + s + "\n);\n\n"

    dbt_path = f"C:\\Sources\\Repos\\EnterpriseCloud\\dbt\\{project}\\models\\{schema.lower()}\\{schema}__{table.upper()}.SQL"
    with open(dbt_path, 'w') as f:
        f.write(dbt)

    snow_path = f"C:\\Sources\\Repos\\EnterpriseCloud\\Snow\\{schema}-{db}.sql"
    with open(snow_path, 'w') as f:
        f.write(snow)

print('schema.yml additions:')
print(yml)
